In [1]:
import torch


In [2]:
import torch.nn as nn


In [3]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [4]:

GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}


In [5]:
class LayerNorm(nn.Module):
  def __init__(self,emb_dim):
    super().__init__()
    self.eps=1e-5
    self.scale=nn.Parameter(torch.ones(emb_dim))
    self.shift=nn.Parameter(torch.zeros(emb_dim))
  def forward(self,x):
    mean=x.mean(dim=-1,keepdim=True)
    var=x.var(dim=-1,keepdim=True,unbiased=True)
    norm_x=(x-mean)/torch.sqrt(var+self.eps)
    return self.scale*norm_x+self.shift

In [6]:
import math

In [7]:
class GELU(nn.Module):
  def __init__(self):
    super().__init__()
  def forward(self, x):
        return 0.5 * x * (
            1 + torch.tanh(
                math.sqrt(2.0 / math.pi) * (x + 0.044715 * x**3)
            )
        )

In [8]:
class FeedForward(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.layers=nn.Sequential(
        nn.Linear(cfg["emb_dim"],4*cfg["emb_dim"]),
        GELU(),
        nn.Linear(4*cfg["emb_dim"],cfg["emb_dim"])
    )
  def forward(self,x):
    return self.layers(x)

In [9]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )
    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec


In [10]:
class TransformerBlock(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.att=MultiHeadAttention(
      d_in=cfg["emb_dim"],
      d_out=cfg["emb_dim"],
      context_length=cfg["context_length"],
      num_heads=cfg["n_heads"],
      dropout=cfg["drop_rate"],
      qkv_bias=cfg["qkv_bias"])
    self.ff=FeedForward(cfg)
    self.norm1=LayerNorm(cfg["emb_dim"])
    self.norm2=LayerNorm(cfg["emb_dim"])
    self.drop_shortcut=nn.Dropout(cfg["drop_rate"])
  def forward(self,x):
    shortcut=x
    x=self.norm1(x)
    x=self.att(x)
    x=self.drop_shortcut(x)
    x+=shortcut
    shortcut=x
    x=self.norm2(x)
    x=self.ff(x)
    x=self.drop_shortcut(x)
    x+=shortcut
    return x

In [11]:
# Token Embedding
# Positional Embedding
# Input Embedding=Token Embedding + Positional Embedding
# Dropout (Randomly turn off embedding elements to zero with a probability of p)
# Transformer

  ## Layer Normalization(with mean =0 variance =1)
  ## Masked Multi head attention to Generate contect vector embedding using attention weight for no of head
  ## Dropout
  ## Shortcut connection(prevents VGD)
  ## Layer Norm
  ## Feed Forward (expnasion and contraction)
  ## Output dimension is same as input
  ## Dropout
  ## Shortcut Connection
  ## Transformer block Output
# Layer Normalisation
# Output Head(Neural layer (emb_dim*vocab size))
# Output Logits for each word we have vocabsize no of token
# Batch_size *cotext size * vocab size




In [12]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [13]:
torch.manual_seed(123)
model=GPTModel(GPT_CONFIG_124M)
batch=torch.tensor(
    [[6109,3626,6100,345],
     [6109,1110,6622,257]]
)
out=model(batch)
print(f"Batch is :{batch}")
print(f"\nOutput shape:{out.shape}")
print(out)

Batch is :tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape:torch.Size([2, 4, 50257])
tensor([[[ 0.3612,  0.4223, -0.0709,  ...,  0.3479,  0.4655, -0.2833],
         [-0.1785, -0.5656, -0.9477,  ...,  0.0476,  0.5173, -0.3160],
         [ 0.7118,  0.0335,  0.1078,  ...,  0.1020, -0.4331, -0.2547],
         [-1.0068,  0.3420, -0.1191,  ...,  0.7193,  0.4018,  0.0532]],

        [[-0.2562,  0.0899,  0.0337,  ...,  0.2659,  0.4448, -0.6800],
         [ 0.1230,  0.3651, -0.2071,  ...,  0.7704,  0.2702,  0.2250],
         [ 1.0555,  1.0312, -0.2797,  ...,  0.6934,  0.3201, -0.3172],
         [-0.1559,  0.3922,  0.3286,  ...,  1.2627, -0.1862,  0.0391]]],
       grad_fn=<UnsafeViewBackward0>)


In [14]:
total_params=sum(p.numel() for p in model.parameters())
print(f"total number of parameters:{total_params:,}")

total number of parameters:163,009,536


In [15]:
def gen_text(model,idx,max_new_tokens,context_size):
  for _ in range(max_new_tokens):
    idx_cond=idx[:,-context_size:]
    with torch.no_grad():
      logits=model(idx_cond)
    logits=logits[:,-1,:]
    probabs=torch.softmax(logits,dim=-1)
    idx_next=torch.argmax(probabs,dim=-1,keepdim=True)
    idx=torch.cat((idx,idx_next),dim=1)
  return idx


In [16]:
torch.manual_seed(123)
start_content="hello,I am"
encoded=tokenizer.encode(start_content)
print("Encoded",encoded)
encoded_tensor=torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor_shape",encoded_tensor.shape)

Encoded [31373, 11, 40, 716]
encoded_tensor_shape torch.Size([1, 4])


In [19]:
model.eval()
out=gen_text(model=model,idx=encoded_tensor,max_new_tokens=10,context_size=GPT_CONFIG_124M["context_length"])
print("Output",out)
print(len(out[0]))

Output tensor([[31373,    11,    40,   716, 27018, 24086, 47843, 30961, 42348, 15635,
         46200,   678, 30595, 48796]])
14


In [20]:
decoded_text=tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

hello,I am Featureiman ByeswickattributeometerSin 19 Rutgersjen


In [21]:
def text_token(text,tokenizer):
  encoded=tokenizer.encode(text,allowed_special={'<|endoftext|>'})
  encoded_tensor=torch.tensor(encoded).unsqueeze(0)
  return encoded_tensor
def token_text(token_ids,tokenizer):
  flat=token_ids.squeeze(0)
  return tokenizer.decode(flat.tolist())
start_content="Every effort moves you"
tokenizer=tiktoken.get_encoding("gpt2")
token_ids=gen_text(
    model=model,
    idx=text_token(start_content,tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)
print("Output text\n",token_text(token_ids,tokenizer))

Output text
 Every effort moves you Aeiman Byeswickattributeometer inspector Normandy freezerigrate


In [22]:
inputs=torch.tensor([[16833,3626,6100],
                     [40,1107,588]])
targets=torch.tensor([[3626,6100,345],
                      [1107,588,11311]])


In [23]:
with torch.no_grad():
  logits=model(inputs)
probabs=torch.softmax(logits,dim=-1)
print(probabs,"\n",probabs.shape)


tensor([[[1.5623e-05, 9.6788e-06, 2.3833e-05,  ..., 1.9601e-05,
          2.7803e-05, 5.2910e-06],
         [2.1814e-05, 7.4753e-06, 8.7612e-06,  ..., 1.1750e-05,
          2.4776e-05, 1.4423e-05],
         [4.0961e-05, 1.2124e-05, 1.9110e-05,  ..., 1.8175e-05,
          1.2299e-05, 1.4953e-05]],

        [[2.8878e-05, 1.2903e-05, 4.2572e-05,  ..., 9.8570e-06,
          3.3994e-05, 7.1146e-06],
         [2.0216e-05, 2.4666e-05, 1.9497e-05,  ..., 6.1105e-06,
          3.2119e-05, 1.3273e-05],
         [3.1136e-05, 1.9669e-05, 2.4852e-05,  ..., 1.2861e-05,
          1.4022e-05, 2.1336e-05]]]) 
 torch.Size([2, 3, 50257])


In [24]:
token_ids=torch.argmax(probabs,dim=-1,keepdim=True)
print("Token IDs:\n",token_ids)

Token IDs:
 tensor([[[36397],
         [39619],
         [20610]],

        [[ 8615],
         [49289],
         [47105]]])


In [27]:
print(f"Targets batch 1:{token_text(targets[0],tokenizer)}")
print(f"Targets batch 1:{token_text(token_ids[0].flatten(),tokenizer)}")

Targets batch 1: effort moves you
Targets batch 1: Gathering SerbianFriday


In [29]:
import torch.nn as nn

In [30]:
text_idx=0
target_probab=probabs[text_idx,[0,1,2],targets[text_idx]]
print("Text 1",target_probab)
text_idx=1
target_probabs=probabs[text_idx,[0,1,2],targets[text_idx]]
print("text 2",target_probabs)

Text 1 tensor([2.3469e-05, 2.0536e-05, 1.1741e-05])
text 2 tensor([4.2778e-05, 1.6253e-05, 1.1592e-05])


In [31]:
log_probabs=torch.log(torch.cat((target_probab,target_probabs)))
print(log_probabs)

tensor([-10.6598, -10.7933, -11.3524, -10.0595, -11.0272, -11.3652])


In [32]:
avg_log=torch.mean(log_probabs)
print(avg_log)

tensor(-10.8762)


In [33]:
pos_log=avg_log*-1

In [34]:
logits_flat=logits.flatten(0,1)
targets_flat=targets.flatten()
print(logits_flat.shape,targets_flat.shape)

torch.Size([6, 50257]) torch.Size([6])


In [35]:
loss=nn.functional.cross_entropy(logits_flat,targets_flat)
print(loss)

tensor(10.8762)


In [37]:
perplexity=torch.exp(loss)
perplexity

tensor(52904.6484)